In [0]:
from pyspark.sql.functions import *

# Read bronze
df = spark.table("bronze.payers")


# fixing column
df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])

# Dropping unwanted columns
df = df.drop("_line", "_fivetran_synced")

# Trim all string columns
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

# Handling nulls
df = df.fillna({
    "name": "unknown",
    "state_headquartered": "unknown",
    "city": "unknown"
})

# Fix zip datatype
df = df.withColumn("zip", col("zip").cast("int"))

# Remove bad rows
df = df.filter(col("id").isNotNull())

# Replace 'unknown' with NULL (case-insensitive)
df = df.select([
    when(lower(col(c)) == "unknown", None).otherwise(col(c)).alias(c)
    for c in df.columns
])

# Remove system columns (like _line, _fivetran_synced)
df = df.select([
    col(c) for c in df.columns if not c.startswith("_")
])

# Save to silver
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.payers")